# Chapter 15 — Stress–Strain Relationships
### *Modern Strain Gage Technology* — Companion Notebook

**A strain gage reads deformation, but the engineer wants stress — and the entire bridge between the two, for an isotropic elastic solid, is just two numbers: Young's modulus *E* and Poisson's ratio *ν*.**

Companion notebook to Chapter 15 — Stress–Strain Relationships.

The relationship between stress and strain is where material science meets measurement. For an isotropic linear elastic material, six independent stress components produce six independent strain components through a 6×6 compliance matrix — but because of symmetry and isotropy, the entire matrix is determined by just two constants: Young's modulus *E* and Poisson's ratio *ν*. Every other elastic constant (shear modulus *G*, bulk modulus *K*, the Lamé constants) follows algebraically.

This matters for strain gage practice in two ways. First, a gage reads strain — but engineers usually want stress, and the conversion requires knowing the elastic constants of the specimen material. Second, the choice between **plane stress** and **plane strain** assumptions changes the computed stress from the same strain reading; getting that choice wrong can introduce errors of 10–20% in the inferred stress even when the measured strain is perfectly accurate.

**This notebook demonstrates:**
1. The 3D compliance matrix and derived elastic constants for a user-specified *E* and *ν*
2. Young's modulus and shear modulus compared across seven structural materials
3. A numerical comparison of plane stress vs. plane strain strain components for the same applied load

In [ ]:
# Colab setup: install dependencies (uncomment on a fresh Colab runtime)
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# All figures generated here are written to this directory; the book includes
# these exact files, so do not change the output path or filenames.
OUT_DIR = Path("../src/media/ch15")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — The Compliance Matrix (Isotropic Material)

The compliance matrix **S** encodes the complete linear elastic behavior of an isotropic material: strain equals compliance times stress, {ε} = [S]{σ}. For an isotropic solid, the matrix has a characteristic block structure — the normal-stress terms are coupled by ν (Poisson contraction), while the shear terms are completely decoupled and governed by G = E / 2(1+ν).

The worked example below applies 100 MPa of uniaxial tension in the x-direction and shows all six resulting strain components. Two things to notice: ε_x is positive (extension in the load direction) while ε_y and ε_z are negative and equal to −ν·ε_x (Poisson contraction). All three shear strains are zero, as expected for a principal load direction. This is the calculation a strain gage analyst performs in reverse — given measured ε_x, ε_y from a two-element rosette on a free surface, recover σ_x and σ_y.

In [ ]:
# --- Material properties (edit these) ---
E_GPa = 200.0   # Young's modulus, GPa
NU    = 0.29    # Poisson's ratio (dimensionless)

E_MPa = E_GPa * 1e3  # work in MPa, so stresses in MPa yield strains directly


def elastic_constants(E, nu):
    """Return the derived isotropic elastic constants (G, K, lambda) from (E, nu).

    G is the shear modulus, K the bulk modulus, and lambda Lame's first
    constant. All three share the same pressure units as the input E.
    """
    G = E / (2 * (1 + nu))
    K = E / (3 * (1 - 2 * nu))
    lame_lambda = E * nu / ((1 + nu) * (1 - 2 * nu))
    return G, K, lame_lambda


def compliance_matrix(E, nu):
    """Build the 6x6 isotropic compliance matrix S, where {eps} = S @ {sigma}.

    Voigt ordering is [xx, yy, zz, xy, xz, yz]. Normal-stress terms are
    coupled by nu (Poisson contraction); the three shear rows are decoupled
    and governed by G = E / (2(1 + nu)), using engineering shear strain.
    """
    G = E / (2 * (1 + nu))
    return np.array([
        [ 1/E,  -nu/E, -nu/E,   0,   0,   0],
        [-nu/E,  1/E,  -nu/E,   0,   0,   0],
        [-nu/E, -nu/E,  1/E,    0,   0,   0],
        [   0,     0,     0,  1/G,   0,   0],
        [   0,     0,     0,    0, 1/G,   0],
        [   0,     0,     0,    0,   0, 1/G],
    ])


G, K, lame_lambda = elastic_constants(E_MPa, NU)
print(f"Input:  E = {E_GPa} GPa,  ν = {NU}")
print(f"Derived: G = {G/1e3:.2f} GPa  (shear modulus)")
print(f"         K = {K/1e3:.2f} GPa  (bulk modulus)")
print(f"         λ = {lame_lambda/1e3:.2f} GPa  (Lamé 1st constant)")
print()

S = compliance_matrix(E_MPa, NU)

# Worked example: uniaxial tension applied along x
UNIAXIAL_STRESS_MPa = 100.0
sigma_vec = np.array([UNIAXIAL_STRESS_MPa, 0, 0, 0, 0, 0])  # [σx,σy,σz,τxy,τxz,τyz]
eps_vec = S @ sigma_vec
labels = ['ε_x', 'ε_y', 'ε_z', 'γ_xy', 'γ_xz', 'γ_yz']
print(f"Uniaxial tension σ_x = {UNIAXIAL_STRESS_MPa:.0f} MPa:")
for lbl, val in zip(labels, eps_vec):
    print(f"  {lbl:6s} = {val*1e6:8.2f} με")

---
## 2 — Material Property Comparison

Not all structural materials are created equal in the context of strain measurement. A gage bonded to steel (E = 200 GPa) produces a resistance change for a given load that is nearly three times smaller than the same gage on aluminum (E = 70 GPa) under the same stress — because the strain is three times larger in the softer material. This has direct implications for signal-to-noise ratio, gage selection, and excitation voltage.

The shear modulus G = E / 2(1+ν) tells a parallel story. Materials with similar E but different ν (steel vs. Inconel, for example) will have different shear moduli — and different sensitivities when gages are mounted in a torsion configuration. The bar chart below lets you compare both constants across the materials most commonly encountered in structural testing.

In [ ]:
# Representative structural materials: (E [GPa], Poisson's ratio, density [kg/m³])
materials = {
    'Steel 4340':         (200, 0.29, 7800),
    'Aluminum 7075':      ( 71, 0.33, 2810),
    'Titanium Ti-6Al-4V': (114, 0.34, 4430),
    'Copper C110':        (120, 0.34, 8940),
    'Inconel 718':        (200, 0.29, 8200),
    'CFRP (0° ply)':      (135, 0.28, 1600),
    'Glass/epoxy':        ( 45, 0.28, 2100),
}
names = list(materials.keys())
E_list  = [props[0] for props in materials.values()]
nu_list = [props[1] for props in materials.values()]
# Shear modulus G = E / (2(1 + nu)), computed per material (GPa)
G_list = [E_i / (2 * (1 + nu_i)) for E_i, nu_i in zip(E_list, nu_list)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(names, E_list, color='steelblue', alpha=0.8)
axes[0].set_xlabel('E (GPa)')
axes[0].set_title("Young's Modulus")
axes[0].grid(True, axis='x', linestyle=':', alpha=0.5)

axes[1].barh(names, G_list, color='tomato', alpha=0.8)
axes[1].set_xlabel('G (GPa)')
axes[1].set_title('Shear Modulus')
axes[1].grid(True, axis='x', linestyle=':', alpha=0.5)

fig.tight_layout()
fig.savefig(OUT_DIR / 'fig15_nb1_modulus_comparison_materials.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 3 — Plane Stress vs. Plane Strain

Surface strain gages always operate in a *plane stress* condition: because the free surface cannot carry a normal traction, σ_z = 0 at the surface where the gage is bonded. The three in-plane strain components are free to take whatever values the applied loads dictate, and the out-of-plane strain ε_z is nonzero (though unmeasured by the gage).

Contrast this with *plane strain*, which applies in the interior of a long structure constrained against axial deformation. There, ε_z = 0 by constraint, and σ_z is nonzero. Using a plane strain constitutive relation to interpret surface gage data — or vice versa — introduces systematic error in every computed stress component. The printed comparison below makes the magnitude of that error concrete for a typical biaxial stress state.

In [ ]:
# Compare plane stress vs. plane strain for the SAME applied in-plane stresses.
#   Plane stress: σ_z = 0  → ε_z ≠ 0   (thin plates, free surfaces — where gages live)
#   Plane strain: ε_z = 0  → σ_z ≠ 0   (long or thick laterally constrained bodies)
E_STEEL_MPa = 200e3   # Young's modulus, MPa
NU_STEEL    = 0.29    # Poisson's ratio

# Applied in-plane stress state (MPa)
SIGMA_X = 100.0
SIGMA_Y = 50.0
TAU_XY  = 20.0


def plane_stress_strains(sx, sy, E, nu):
    """Normal strains for a plane-stress state (σ_z = 0); returns (ε_x, ε_y, ε_z).

    The out-of-plane strain ε_z is generally nonzero even though σ_z = 0.
    """
    ex = (sx - nu * sy) / E
    ey = (sy - nu * sx) / E
    ez = -nu / E * (sx + sy)
    return ex, ey, ez


def plane_strain_strains(sx, sy, E, nu):
    """Normal strains for a plane-strain state (ε_z = 0); returns (ε_x, ε_y, ε_z).

    Out-of-plane deformation is constrained, so ε_z is identically zero while
    an out-of-plane stress σ_z = nu(sx + sy) develops (not returned here).
    """
    ex = (1/E) * ((1 - nu**2) * sx - nu * (1 + nu) * sy)
    ey = (1/E) * ((1 - nu**2) * sy - nu * (1 + nu) * sx)
    ez = 0.0
    return ex, ey, ez


ex_ps, ey_ps, ez_ps = plane_stress_strains(SIGMA_X, SIGMA_Y, E_STEEL_MPa, NU_STEEL)
ex_pe, ey_pe, ez_pe = plane_strain_strains(SIGMA_X, SIGMA_Y, E_STEEL_MPa, NU_STEEL)

print(f"In-plane stresses: sx={SIGMA_X:.0f}, sy={SIGMA_Y:.0f}, txy={TAU_XY:.0f} MPa")
print(f"{'':20s} {'Plane stress':>14} {'Plane strain':>14}")
print(f"{'ε_x (με)':20s} {ex_ps*1e6:>14.2f} {ex_pe*1e6:>14.2f}")
print(f"{'ε_y (με)':20s} {ey_ps*1e6:>14.2f} {ey_pe*1e6:>14.2f}")
print(f"{'ε_z (με)':20s} {ez_ps*1e6:>14.2f} {ez_pe*1e6:>14.2f}")
print()
print("Strain gages on free surfaces measure plane stress conditions.")
print("The out-of-plane strain ε_z is not zero, but cannot be measured by a surface gage.")

---
## Where to take this next

These three cells are a starting point. A few concrete extensions:

- **Turn the plane-stress equations into a rosette solver.** Take three measured strains from a 0°/45°/90° rectangular rosette, compute the principal strains and their orientation, then apply the strain-to-stress relations (Eqs. 15.11–15.13) to recover σ_x, σ_y, and τ_xy. Check the result against a hand calculation for a known load.
- **Propagate uncertainty in *E* and *ν*.** The chapter warns that errors in the elastic constants feed straight into inferred stress. Wrap `plane_stress_strains` in a Monte Carlo loop — seed the generator with `np.random.default_rng(0)`, perturb *E* and *ν* within their datasheet tolerances, and plot the resulting spread in computed stress.
- **Quantify the plane-stress vs. plane-strain gap.** Sweep Poisson's ratio from 0.25 to 0.49 and plot the percentage difference between the two ε_x predictions for a fixed stress state — making concrete why the choice matters most for nearly incompressible materials.